## সূচিপত্র

- [এই সপ্তাহের Math কেন দরকার](#s1)

**Part 1: AI Theory**
- [Word/Text Embeddings](#s2)
- [Embedding কী](#s3)
- [Semantic Similarity: Semantic vs Keyword Search](#s4)
- [দুই রকম search](#s5)
- [Dot Product ও Cosine Similarity](#s6)
- [Dot Product](#s7)
- [Cosine Similarity](#s8)
- [Attention ও Self-Attention](#s9)
- [Attention](#s10)
- [হিসাবটা কীভাবে হয়](#s10)
- [Transformer Architecture](#s11)
- [Transformer](#s12)
- [Positional Encoding](#s13)
- [কেন দরকার](#s14)

**Part 2: Math Track, Matrices**
- [Matrix, Row, Column ও Shape](#s15)
- [Matrix](#s16)
- [Addition, Subtraction ও Scalar Multiplication](#s17)
- [তিনটা কাজ](#s18)
- [Transpose](#s22)
- [Matrix Multiplication](#s19)
- [কীভাবে হয়](#s20)
- [Matrix-Vector Multiplication](#s23)
- [Shape Rule](#s24)
- [Week-end Gate](#s21)

<a id="s1"></a>

#### এই সপ্তাহের Math কেন দরকার

Embedding হলো সংখ্যার একটা টেবিল (Matrix) থেকে একটা সারি তুলে আনা, আর Attention-এর হিসাব হলো Matrix-এর গুণ; তাই Matrix না বুঝলে Embedding ও Attention বোঝা যায় না।

#### Part 1: AI Theory

<a id="s2"></a>

#### Word/Text Embeddings

<a id="s3"></a>

#### আগে একটা গল্প
লাইব্রেরিতে বই এলোমেলো পড়ে থাকলে খোঁজা কঠিন। তাই লাইব্রেরিয়ান বিষয় ধরে বই সাজান: গল্পের বই এক তাকে, রান্নার বই আরেক তাকে। "মিষ্টি বানানোর বই" থাকে রান্নার তাকের একদম পাশে। বইটা কোথায় বসানো আছে, সেটা দেখেই বোঝা যায় কোন বই কোন বইয়ের কাছাকাছি।

#### Embedding কী
Week 02-এ দেখেছ, tokenizer প্রতিটা token-কে একটা নম্বর (token ID) দেয়। কিন্তু ওই নম্বরে অর্থ নেই। ৫ নম্বর আর ৬ নম্বর token পাশাপাশি আছে বলেই তাদের অর্থ কাছাকাছি, তা নয়।

অর্থ ধরার জন্য প্রতিটা token-কে সংখ্যার একটা তালিকা (Vector) দেওয়া হয়, যার নাম **embedding**। লাইব্রেরির তাকে বইয়ের জায়গার মতো, এই Vector দিয়েই শব্দের "জায়গা" ঠিক হয়। অর্থ কাছাকাছি হলে Vector-ও কাছাকাছি হয়, যেমন "রাজা" আর "রানি"। "আম" থাকে তাদের থেকে অনেক দূরে। এই সংখ্যাগুলো কেউ হাতে লেখে না, model প্রচুর লেখা পড়ে নিজে শেখে।

শুধু শব্দ নয়, পুরো বাক্য বা অনুচ্ছেদের জন্যও একটা Vector বানানো যায়। তাকে বলে **text embedding**। পরের topic-এর search এই Vector দিয়েই চলে।

সব শব্দের Vector একটা টেবিলে সাজানো থাকে, যেখানে প্রতিটা সারি একটা শব্দ। এই টেবিলটাই একটা **Matrix**, আর token ID দিয়ে সেই টেবিল থেকে একটা সারি তুলে আনাই হলো embedding পাওয়া। Matrix নিয়ে Part 2-তে পড়ব।

**Demo:** নিচের code-এ তিনটা শব্দের একটা ছোট embedding টেবিল আছে। সংখ্যাগুলো বোঝার জন্য হাতে বানানো, আসল model-এ একেকটা Vector-এ কয়েকশো সংখ্যা থাকে।

#### মনে রাখার এক লাইন
> Embedding হলো token-এর অর্থকে সংখ্যার Vector বানানো, যাতে কাছাকাছি অর্থের শব্দের Vector কাছাকাছি থাকে।

In [1]:
import numpy as np

words = ["রাজা", "রানি", "আম"]

E = np.array([[0.8, 0.7, 0.1],
              [0.7, 0.8, 0.1],
              [0.1, 0.0, 0.9]])

print("টেবিলের shape:", E.shape)

token_id = words.index("রানি")
print("token ID:", token_id)

print("রানি-র embedding:", E[token_id])

টেবিলের shape: (3, 3)
token ID: 1
রানি-র embedding: [0.7 0.8 0.1]


<a id="s4"></a>

#### Semantic Similarity: Semantic vs Keyword Search

<a id="s5"></a>

#### আগে একটা গল্প
ওষুধের দোকানে গিয়ে তুমি বললে, "মাথাব্যথার ওষুধ দিন।" দোকানদার তুমি কী চাও তা বুঝলেন, আর ব্যথা কমানোর ওষুধ বের করে দিলেন। অন্য এক দোকানদার শুধু প্যাকেটের গায়ে "মাথাব্যথা" শব্দটা মেলাত। প্যাকেটে অন্য শব্দ লেখা থাকলে সে কিছুই খুঁজে পেত না, অথচ ওষুধ তাকেই ছিল।

#### দুই রকম search
**Keyword search** প্রশ্নের শব্দগুলো লেখার মধ্যে হুবহু আছে কি না তাই মেলায়। শব্দ না মিললে সে কিছু পায় না। আবার শব্দ মিললেই ভুল জিনিসও তুলে আনে।

**Semantic search** প্রশ্ন আর প্রতিটা লেখার embedding বানায়, তারপর দেখে কার অর্থ প্রশ্নের সবচেয়ে কাছে। শব্দ না মিললেও অর্থ মিললে সে ঠিক লেখাটা পায়। এই "অর্থের কাছাকাছি থাকা"-কেই বলে **semantic similarity**।

ধরো প্রশ্ন "গাড়ি ঠিক করার দোকান"। একটা লেখা "মোটর গ্যারেজে যন্ত্র সারানো হয়", যেটা ঠিক উত্তর কিন্তু একটা শব্দও মেলে না। আরেকটা লেখা "গাড়ি কেনাবেচার দোকান", যেটা ভুল উত্তর কিন্তু দুটো শব্দ মিলে যায়। Keyword search ভুলটাকে বেছে নেয়।

**Demo:** নিচের code-এ keyword search এই ভুলটা করে দেখাচ্ছে। Semantic search কীভাবে ঠিকটা বাছে, তা W04 Math-এ cosine similarity হাতে-কলমে করে দেখব।

#### মনে রাখার এক লাইন
> Keyword search শব্দ মেলায়, আর semantic search অর্থ মেলায়।

In [2]:
query = "গাড়ি ঠিক করার দোকান"
docs = [
    "মোটর গ্যারেজে যন্ত্র সারানো হয়",
    "গাড়ি কেনাবেচার দোকান",
]

query_words = set(query.split())

for doc in docs:
    common = query_words & set(doc.split())
    print(len(common), "টা শব্দ মিলেছে:", doc)

0 টা শব্দ মিলেছে: মোটর গ্যারেজে যন্ত্র সারানো হয়
2 টা শব্দ মিলেছে: গাড়ি কেনাবেচার দোকান


<a id="s6"></a>

#### Dot Product ও Cosine Similarity

<a id="s7"></a>

#### আগে একটা গল্প
দুই বন্ধু ঠিক করছে সন্ধ্যায় কী করবে। ক্রিকেট, গান আর রান্না, এই তিন বিষয়ে দুজনেই ০ থেকে ১০-এর মধ্যে নিজের পছন্দের নম্বর দিল। প্রতি বিষয়ে দুজনের নম্বর গুণ করে সব গুণফল যোগ করলে একটা মোট সংখ্যা পাওয়া যায়। যেখানে দুজনেরই নম্বর বেশি, সেখানে গুণফল বড় হয়। যেখানে একজন ভালোবাসে আর অন্যজন ভালোবাসে না, সেখানে গুণফল ছোট হয়। তাই মোট সংখ্যা বড় মানে দুজনের পছন্দ বেশি মেলে।

#### Dot Product
**Dot product** হলো দুটো Vector-এর একই জায়গার সংখ্যাগুলো গুণ করে সব গুণফল যোগ করা।

> [3, 4] এবং [4, 2] হলে: 3×4 + 4×2 = 12 + 8 = 20

এর ফল একটা সংখ্যা, Vector নয়। যত বেশি জায়গায় দুই Vector-এর সংখ্যা মেলে, ফল তত বড় হয়।

**Demo ১:** আগের embedding-এর তিনটা শব্দই এখানে নিয়েছি: a হলো "রাজা", b হলো "রানি", c হলো "আম"। রাজা-রানির dot product বড় আর রাজা-আমের ছোট আসার কথা।

In [3]:
import numpy as np

a = np.array([0.8, 0.7, 0.1])
b = np.array([0.7, 0.8, 0.1])
c = np.array([0.1, 0.0, 0.9])

print(np.dot(a, b))  # Dot product of a and b
print(np.dot(a, c))  # Dot product of a and c

1.13
0.17000000000000004


**Demo ২:** `np.dot` ছাড়াও `@` চিহ্ন দিয়ে dot product করা যায়। আর দুই Vector-এর ক্রম বদলালেও ফল একই থাকে।

In [4]:
A = np.array([3,4])
B = np.array([4, 2])

print(A @  B)
print(B @ A)

20
20


<a id="s8"></a>

#### Cosine Similarity
Dot product-এর একটা দুর্বলতা আছে। Vector-এর সংখ্যা বড় হলে অর্থ না মিললেও গুণফল এমনিতেই বড় হয়ে যায়। ধরো এক বন্ধু সব বিষয়েই ১০ নম্বর দেয়, তার সাথে সবার মোট সংখ্যাই বড় আসবে।

তাই অনেক সময় Vector কতটা বড়, সেটা না দেখে শুধু Vector-গুলো কোন **দিকে** মুখ করে আছে, সেটাই তুলনা করা হয়। দুজন মানুষ একই দিকে আঙুল তুললে একজনের হাত লম্বা আর একজনের ছোট হলেও দুজনের দিক একই। এই দিকের মিল মাপার পদ্ধতির নাম **cosine similarity**। এর মান সবসময় −১ থেকে ১-এর মধ্যে থাকে।

| মান | মানে |
|---|---|
| ১-এর কাছে | দুই Vector প্রায় একই দিকে, অর্থ খুব কাছাকাছি |
| ০-এর কাছে | দুটোর মধ্যে কোনো সম্পর্ক নেই |
| −১-এর কাছে | দুটো উল্টো দিকে |

Cosine similarity-র সূত্র ও code W04 Math-এ হাতে-কলমে করা হবে।

#### মনে রাখার এক লাইন
> Dot product মেলার পরিমাণ মাপে, কিন্তু Vector-এর সংখ্যা বড় হলে এমনিতেই বড় আসে। Cosine similarity শুধু দিকের মিল মাপে, তাই Embedding তুলনায় সাধারণত সেটাই ব্যবহার হয়।

<a id="s9"></a>

#### Attention ও Self-Attention

<a id="s10"></a>

#### আগে একটা গল্প
আড্ডায় কেউ বলল, "রহিম করিমকে বলল, সে কাল আসবে।" "সে" শুনেই তুমি মনে মনে আগের কথাগুলোর দিকে তাকাও আর ভাবো: কার কথা বলা হলো? তুমি আগের সব শব্দের দিকে সমান নজর দাও না। "রহিম" আর "করিম"-এর দিকে বেশি মন দাও, "বলল" বা "কাল"-এর দিকে কম। নজর বেছে বেছে দেওয়ার এই ক্ষমতাই attention।

#### Attention
Model যখন একটা শব্দ বোঝে, তখন বাক্যের অন্য প্রতিটা শব্দকে একটা গুরুত্বের নম্বর দেয়, যার নাম **attention weight**। যে শব্দের নম্বর বেশি, তার অর্থ ওই শব্দের নতুন অর্থে বেশি করে মেশে।

যখন একটা বাক্যের শব্দেরা একই বাক্যের শব্দগুলোর দিকেই তাকায়, তখন তাকে বলে **self-attention**। "সে" শব্দটা ঠিক এভাবেই "রহিম"-এর দিকে ঝুঁকে নিজের অর্থ ঠিক করে নেয়।

#### হিসাবটা কীভাবে হয়
তিন ধাপে:

১. "সে"-র Vector আর বাক্যের প্রতিটা শব্দের Vector-এর **dot product** বের করা হয়। এটাই score, অর্থাৎ কে কতটা মেলে।

২. Score-গুলোকে **softmax**-এ ফেলা হয়। Week 02-এর Temperature-এ এটা দেখেছ। এতে score-গুলো এমন নম্বর হয়ে যায় যাদের যোগফল ১।

৩. ওই নম্বর দিয়ে সব শব্দের Vector-এর **ওজনযুক্ত যোগফল** নেওয়া হয়। এটাই "সে"-র নতুন Vector। এতে যার নম্বর বেশি, তার অর্থ বেশি মিশে থাকে।

> **নোট:** আসল Transformer-এ score-কে softmax-এ দেওয়ার আগে √d দিয়ে ভাগ করে ছোট করা হয় (d হলো Vector-এর dimension)। এখানে সরল রাখা হয়েছে।

**Demo:** নিচের code-এ ছয়টা শব্দের Vector হাতে বানানো, সংখ্যা তিনটা হলো মানুষ-ভাব, কাজ-ভাব ও সময়-ভাব। সংখ্যাগুলো এমন দেওয়া হয়েছে যাতে "সে" রহিমের দিকে ঝোঁকে। আসল model এই সংখ্যা নিজে শেখে, আর সেখানে প্রতিটা শব্দের Query, Key ও Value নামে তিনটা আলাদা রূপ থাকে। এখানে বোঝার সুবিধার জন্য একটাই Vector ধরা হয়েছে।

#### মনে রাখার এক লাইন
> Self-attention হলো বাক্যের প্রতিটা শব্দ বাকি শব্দগুলোকে গুরুত্বের নম্বর দিয়ে নিজের অর্থ ঠিক করা।

In [6]:
words = ["রহিম", "করিমকে", "বলল", "সে", "কাল", "আসবে"]

X = np.array([[3.0, 0.0, 0.0],
              [1.5, 0.0, 0.0],
              [0.0, 3.0, 0.0],
              [2.0, 0.5, 0.0],
              [0.0, 0.0, 3.0],
              [0.0, 2.0, 2.0]])

def softmax(scores):
    exp = np.exp(scores - scores.max())
    return exp / exp.sum()

scores = X @ X[3]

weights = softmax(scores)
for word, weight in zip(words, weights):
    print(f"{word:8s} {weight:.2f}")

print("সে-র নতুন Vector:", (weights @ X).round(2))

print("সব শব্দের score-এর shape:", (X @ X.T).shape)

রহিম     0.80
করিমকে   0.04
বলল      0.01
সে       0.14
কাল      0.00
আসবে     0.01
সে-র নতুন Vector: [2.75 0.11 0.02]
সব শব্দের score-এর shape: (6, 6)


<a id="s11"></a>

#### Transformer Architecture

<a id="s12"></a>

#### আগে একটা গল্প
একটা ছোট দল বসেছে একটা বাক্য বুঝতে। প্রতিজনের হাতে একটা করে শব্দ। এক জন এক জন করে পড়ে পরের জনকে বলে না। বরং সবাই একসাথে সবার দিকে তাকিয়ে ঠিক করে, "আমার শব্দের অর্থ আসলে কী, আশপাশের শব্দ দেখে?" এই আলোচনা একবার নয়, কয়েক দফা চলে। প্রতি দফায় সবার বোঝাপড়া আরও পরিষ্কার হয়।

#### Transformer
**Transformer** হলো এমন একটা গঠন (architecture), যার মূল কাজ self-attention। ২০১৭ সালে এটা প্রকাশ হয়েছে, আর আজকের প্রায় সব বড় language model এর ওপর দাঁড়িয়ে। এর সবচেয়ে বড় সুবিধা হলো সে বাক্যের সব শব্দ একসাথে দেখে, তাই কম্পিউটারে দ্রুত চালানো যায়।

সংক্ষেপে এর ধাপগুলো এই:

১. লেখা ভেঙে token হয়, প্রতিটা token-এর embedding (Vector) নেওয়া হয়।

২. প্রতিটা Vector-এর সাথে শব্দটা বাক্যের কোথায় আছে, সেই তথ্য যোগ করা হয়। এর নাম **Positional Encoding**, পরের topic-এ দেখব।

৩. **Self-attention** চলে: প্রতিটা শব্দ বাকি শব্দের দিকে তাকিয়ে নিজের Vector নতুন করে বানায়।

৪. এরপর আরেকটা হিসাবের ধাপ প্রতিটা Vector-কে আলাদা আলাদা করে আরও গুছিয়ে দেয়। এর ভেতরের কথা পরের সপ্তাহগুলোতে পড়ব।

৫. ৩ আর ৪ নম্বর ধাপ মিলে একটা **layer**। এমন অনেকগুলো layer একটার পর একটা সাজানো থাকে, ঠিক গল্পের কয়েক দফা আলোচনার মতো।

৬. সবার শেষের Vector থেকে model পরের token কোনটা হবে, তার সম্ভাবনা বের করে। এখান থেকে Week 02-এর Sampling-এর কাজ শুরু।

মূল Transformer-এ দুটো অংশ ছিল: লেখা বোঝার জন্য encoder আর লেখা বানানোর জন্য decoder। আজকের GPT-ধরনের chat model মূলত decoder অংশই ব্যবহার করে।

#### মনে রাখার এক লাইন
> Transformer হলো self-attention-এর layer-কে পরপর সাজানো একটা গঠন, যেটা বাক্যের সব শব্দ একসাথে দেখে।

<a id="s13"></a>

#### Positional Encoding

<a id="s14"></a>

#### আগে একটা গল্প
একটা বইয়ের সব পাতা খুলে মেঝেতে ছড়িয়ে দিলে সব শব্দই আছে, কিন্তু গল্পটা আর নেই। প্রতিটা পাতায় যদি নম্বর লেখা থাকে, তাহলে আবার ঠিক ক্রমে সাজিয়ে গল্প ফিরে পাওয়া যায়।

#### কেন দরকার
Self-attention সব শব্দ একসাথে দেখে, কিন্তু শব্দ কোনটার আগে বা পরে, তা নিজে থেকে জানে না। অথচ ক্রম বদলালে অর্থ বদলে যায়। "রহিম আগে এল, করিম পরে" আর "করিম আগে এল, রহিম পরে" বাক্য দুটোতে শব্দ একই, কিন্তু অর্থ উল্টো।

তাই প্রতিটা শব্দের embedding-এর সাথে তার position-এর (০ নম্বর, ১ নম্বর, ২ নম্বর...) একটা Vector যোগ করে দেওয়া হয়। একে বলে **Positional Encoding**। এতে একই শব্দ আলাদা জায়গায় বসলে তার Vector একটু আলাদা হয়ে যায়, আর model ক্রম বুঝতে পারে।

মূল Transformer এই position Vector বানাতে sin ও cos ব্যবহার করে, যাতে প্রতিটা position-এর সংখ্যার তালিকা আলাদা হয়। আজকের অনেক model আরও অন্য পদ্ধতিও ব্যবহার করে, তবে মূল ধারণা একই: শব্দের সাথে জায়গার তথ্য জুড়ে দেওয়া।

**Demo:** নিচের code-এ চারটা position-এর Vector বানানো হয়েছে, তারপর একই শব্দের Vector দুটো আলাদা position-এ বসিয়ে দেখানো হয়েছে।

#### মনে রাখার এক লাইন
> Positional Encoding শব্দের embedding-এর সাথে তার জায়গার তথ্য যোগ করে, যাতে model শব্দের ক্রম বুঝতে পারে।

In [7]:
def positional_encoding(n_positions, dim):
    position = np.arange(n_positions)[:, None]
    index = np.arange(dim)[None, :]
    angle = position / np.power(10000, 2 * (index // 2) / dim)
    pe = np.zeros((n_positions, dim))
    pe[:, 0::2] = np.sin(angle[:, 0::2])
    pe[:, 1::2] = np.cos(angle[:, 1::2])
    return pe

pe = positional_encoding(4, 4)
print(pe.round(2))

word_vec = np.array([0.1, 0.9, 0.0, 0.2])
print("০ নম্বর জায়গায়:", (word_vec + pe[0]).round(2))
print("২ নম্বর জায়গায়:", (word_vec + pe[2]).round(2))

[[ 0.    1.    0.    1.  ]
 [ 0.84  0.54  0.01  1.  ]
 [ 0.91 -0.42  0.02  1.  ]
 [ 0.14 -0.99  0.03  1.  ]]
০ নম্বর জায়গায়: [0.1 1.9 0.  1.2]
২ নম্বর জায়গায়: [1.01 0.48 0.02 1.2 ]


#### Part 2: Math Track, Matrices

<a id="s15"></a>

#### Matrix, Row, Column ও Shape

<a id="s16"></a>

#### আগে একটা গল্প
স্কুলের মার্কশিট দেখেছ? প্রতিটা সারিতে একজন ছাত্র, আর প্রতিটা খাড়া ঘরে একটা বিষয়। তিনজন ছাত্র আর দুটো বিষয় হলে মার্কশিটে ৩টা সারি আর ২টা খাড়া ঘর থাকে। কে কোন বিষয়ে কত পেল, তা বের করতে শুধু সারি আর খাড়া ঘর মিলিয়ে দেখলেই হয়।

#### Matrix
**Matrix** হলো সংখ্যার একটা আয়তাকার টেবিল, ঠিক ওই মার্কশিটের মতো। আড়াআড়ি সারিকে বলে **row**, খাড়া সারিকে বলে **column**। Week 02-এ শেখা Vector মানে একটা সারির সংখ্যার তালিকা, আর Matrix হলো অনেকগুলো Vector-এর স্তূপ।

Matrix-এর আকারকে বলে **shape**, আর লেখা হয় **(m, n)**, মানে m টা row আর n টা column। আগে row, পরে column। তিন row আর দুই column হলে shape (3, 2)।

NumPy-তে `A.shape` দিয়ে shape পাওয়া যায়। একটা row নিতে `A[0]`, একটা column নিতে `A[:, 1]` লিখতে হয়। গোনা শুরু হয় ০ থেকে, আর `:` মানে "সবগুলো"।

#### মনে রাখার এক লাইন
> Matrix হলো সংখ্যার টেবিল, আর shape (m, n) মানে m টা row ও n টা column।

In [8]:
A = np.array(
    [[1, 2, 3],
     [4, 5, 6]]
)

print(np.shape(A))  # Shape of matrix A

(2, 3)


In [9]:
A = np.array([[1, 2, 3],
              [4, 5, 6]])

print("row 0      :", A[0])
print("column 1   :", A[:, 1])
print("১ নম্বর row, ২ নম্বর column:", A[1, 2])

row 0      : [1 2 3]
column 1   : [2 5]
১ নম্বর row, ২ নম্বর column: 6


<a id="s17"></a>

#### Addition, Subtraction ও Scalar Multiplication

<a id="s18"></a>

#### আগে একটা গল্প
দুই ভাই একই ছকের খাতায় মাসের খরচ লেখে। মাস শেষে দুজনের খাতার একই ঘরের সংখ্যা যোগ করলে মোট খরচের খাতা তৈরি হয়। কিন্তু একজনের খাতায় ঘর কম হলে মিলিয়ে যোগ করা যায় না। আবার দাম সবকিছুর তিনগুণ হয়ে গেলে খাতার প্রতিটা ঘর ৩ দিয়ে গুণ করলেই নতুন খাতা পাওয়া যায়।

#### তিনটা কাজ
**যোগ ও বিয়োগ:** দুটো Matrix-এর একই জায়গার সংখ্যা ধরে ধরে যোগ বা বিয়োগ হয়। এর জন্য দুটোর **shape একই** হতে হবে।

**Scalar multiplication:** একটা সাধারণ সংখ্যা (scalar) দিয়ে Matrix-কে গুণ করলে প্রতিটা ঘর সেই সংখ্যা দিয়ে গুণ হয়।

এই তিনটা কাজ মিলিয়েও করা যায়, যেমন `2*C + D`। আগে scalar গুণ হয়, তারপর যোগ।

#### মনে রাখার এক লাইন
> যোগ-বিয়োগ ঘরে ঘরে হয় ও shape একই লাগে, আর scalar গুণ প্রতিটা ঘরকে একই সংখ্যা দিয়ে গুণ করে।

In [10]:
A = np.array([[1, 2],
              [3, 4]])
B = np.array([[5, 6],
              [7, 8]])
C = np.array([[2, 0],
              [1, 3]])
D = np.array([[1, 4],
              [2, 1]])

print(A + B)  # Element-wise addition of A and B
print(A - B)  # Element-wise subtraction of A and B
print(3 * A)  # Scalar multiplication of A

print(2*C + D)

[[ 6  8]
 [10 12]]
[[-4 -4]
 [-4 -4]]
[[ 3  6]
 [ 9 12]]
[[5 4]
 [4 7]]


<a id="s22"></a>

#### Transpose

#### আগে একটা গল্প
মার্কশিটে সারিতে ছাত্র আর খাড়া ঘরে বিষয় আছে। এখন তুমি চাইছ সারিতে বিষয় আর খাড়া ঘরে ছাত্র। কাগজটাকে কাত করে ঘুরিয়ে ধরলেই হলো: যা আগে সারি ছিল, তা এখন খাড়া ঘর, আর যা খাড়া ঘর ছিল, তা সারি।

#### Transpose
**Transpose** হলো Matrix-এর row আর column অদলবদল করা। A-এর transpose লেখা হয় Aᵀ। NumPy-তে লিখতে হয় `A.T`।

এতে shape-ও উল্টে যায়: (m, n) হয়ে যায় (n, m)। নিচের code-এ A-এর shape (2, 3), তার transpose-এর shape (3, 2)।

Transpose দরকার হয় গুণের আগে shape মেলাতে। যেমন Attention-এ `X @ X.T` দিয়ে সব শব্দের score একসাথে বের করা হয়েছিল।

#### মনে রাখার এক লাইন
> Transpose row-কে column আর column-কে row বানায়, আর shape (m, n) থেকে (n, m) হয়ে যায়।

In [11]:
A = np.array([[1, 2, 3],
              [4, 5, 6]])

M = np.array(
    [[1, 0], [2, 5], [3, 4]]
)

print(A.shape)
print(A.T)
print(A.T.shape)

print(M.shape)
print(M.T)
print(M.T.shape)
    

(2, 3)
[[1 4]
 [2 5]
 [3 6]]
(3, 2)
(3, 2)
[[1 2 3]
 [0 5 4]]
(2, 3)


<a id="s19"></a>

#### Matrix Multiplication

<a id="s20"></a>

#### আগে একটা গল্প
শিক্ষক অনেক ছাত্রের চূড়ান্ত নম্বর বের করছেন। প্রতিটা ছাত্রের সারিতে বিষয়ভিত্তিক নম্বর আছে। আবার দুই রকম হিসাব আছে: একটায় সব বিষয়ের গুরুত্ব সমান, অন্যটায় গণিতের গুরুত্ব বেশি। প্রতিটা ছাত্রের সারির সাথে প্রতিটা হিসাবের গুরুত্বের তালিকা মিলিয়ে একটা করে চূড়ান্ত নম্বর হয়। সব ছাত্র আর সব হিসাবের জন্য মিলিয়ে নম্বরের একটা নতুন টেবিল তৈরি হয়। এটাই matrix multiplication।

#### কীভাবে হয়
A @ B করলে ফলের (i, j) নম্বর ঘরের সংখ্যা হলো **A-এর i নম্বর row** আর **B-এর j নম্বর column**-এর dot product। নিচের উদাহরণে P = [[1, 2], [0, 3]] আর Q = [[4, 1], [2, 5]]:

> প্রথম row, প্রথম column: [1, 2] · [4, 2] = 1×4 + 2×2 = 8
> প্রথম row, দ্বিতীয় column: [1, 2] · [1, 5] = 1×1 + 2×5 = 11
> দ্বিতীয় row, প্রথম column: [0, 3] · [4, 2] = 0×4 + 3×2 = 6
> দ্বিতীয় row, দ্বিতীয় column: [0, 3] · [1, 5] = 0×1 + 3×5 = 15

তাই ফল [[8, 11], [6, 15]]। NumPy-তে `np.dot(P, Q)` আর `P @ Q` একই ফল দেয়।

দুটো জিনিস মনে রেখো। প্রথমত, `*` চিহ্ন matrix multiplication নয়, সেটা ঘরে ঘরে গুণ করে। দ্বিতীয়ত, ক্রম বদলালে ফল বদলে যায়, অর্থাৎ **P @ Q আর Q @ P সাধারণত আলাদা**। এটা সংখ্যার গুণের মতো নয়।

#### মনে রাখার এক লাইন
> Matrix multiplication-এ ফলের প্রতিটা ঘর একটা row আর একটা column-এর dot product, আর ক্রম বদলালে ফল বদলে যায়।

In [12]:
P = np.array([[1, 2], [0, 3]])

Q = np.array([[4, 1], [2, 5]])

print(np.dot(P, Q))
print(P @ Q)  # Using the @ operator for matrix multiplication

[[ 8 11]
 [ 6 15]]
[[ 8 11]
 [ 6 15]]


In [13]:
print(Q @ P)
print(P * Q)

[[ 4 11]
 [ 2 19]]
[[ 4  2]
 [ 0 15]]


<a id="s23"></a>

#### Matrix-Vector Multiplication

#### আগে একটা গল্প
তিনটা দোকানে আম আর কলার দাম আলাদা। দামের তালিকা একটা টেবিলে লেখা: প্রতিটা সারিতে একটা দোকান, প্রথম খাড়া ঘরে আমের দাম আর দ্বিতীয়টায় কলার দাম। তুমি ১০টা আম আর ১টা কলা কিনবে। প্রতিটা দোকানে মোট কত লাগবে? প্রতি দোকানের সারির সাথে তোমার কেনার তালিকা [১০, ১] মিলিয়ে গুণ করে যোগ করলেই পাওয়া যায়।

#### Matrix-Vector Multiplication
Matrix-কে একটা Vector দিয়ে গুণ করলে প্রতিটা row-এর সাথে ওই Vector-এর dot product হয়। ফলে আবার একটা Vector পাওয়া যায়, আর তার সংখ্যা ততটা, Matrix-এ যতটা row আছে।

নিচের উদাহরণে W-এর shape (3, 2) আর x-এর shape (2,)। প্রথম দোকানের জন্য 10×1 + 1×2 = 12। তিন দোকানে মিলে ফল [12, 34, 56], অর্থাৎ shape (3,)।

Embedding আর Attention-এ এই কাজই বারবার চলে: একটা Matrix একটা Vector-কে নতুন Vector বানিয়ে দেয়।

#### মনে রাখার এক লাইন
> Matrix-Vector multiplication-এ প্রতিটা row-এর সাথে Vector-এর dot product হয়, আর ফল একটা নতুন Vector।

In [14]:
W = np.array([[1, 2],
              [3, 4],
              [5, 6]])   # shape (3, 2)
x = np.array([10, 1])

print(W.shape)
print(x.shape)
print(W @ x)


(3, 2)
(2,)
[12 34 56]


<a id="s24"></a>

#### Shape Rule

#### আগে একটা গল্প
প্লাগ আর সকেটের কথা ভাবো। প্লাগে তিনটা পিন থাকলে সকেটেও তিনটা গর্ত থাকতে হবে। পিন আর গর্তের সংখ্যা না মিললে প্লাগ লাগেই না। মিললে তবেই বিদ্যুৎ চলে।

#### Shape Rule
দুটো Matrix গুণ করতে হলে:

> (m, n) @ (n, p) → (m, p)

**মাঝের দুই সংখ্যা মিলতে হবে**, আর ফলের shape হবে বাইরের দুই সংখ্যা। কেন? কারণ A-এর প্রতিটা row-এর দৈর্ঘ্য n আর B-এর প্রতিটা column-এর দৈর্ঘ্য n, আর dot product করতে দুই Vector-এর দৈর্ঘ্য সমান লাগে।

যেমন (2, 3) @ (3, 2) চলবে, ফল (2, 2)। কিন্তু (2, 3) @ (2, 3) চলবে না, কারণ মাঝের ৩ আর ২ মেলে না। NumPy তখন error দেয়।

গুণ করার আগে shape মিলিয়ে নেওয়া ভালো অভ্যাস, কারণ AI-র code-এ সবচেয়ে বেশি ভুল হয় shape মেলাতে গিয়েই।

#### মনে রাখার এক লাইন
> (m, n) @ (n, p) → (m, p): মাঝের দুই সংখ্যা মিলতে হবে, আর ফলের shape বাইরের দুই সংখ্যা।

In [15]:
A = np.array([[1, 2, 3],
              [4, 5, 6]])      # shape (2, 3)
M = np.array([[1, 0],
              [2, 5],
              [3, 4]])         # shape (3, 2)

print((A @ M).shape)
print((M @ A).shape)

try:
    A @ A
except ValueError as error:
    print("Error:", error)

(2, 2)
(3, 3)
Error: matmul: Input operand 1 has a mismatch in its core dimension 0, with gufunc signature (n?,k),(k,m?)->(n?,m?) (size 2 is different from 3)


<a id="s21"></a>

#### Week-end Gate

নিজে উত্তর ভেবে তারপর নিচের উত্তর মেলাও।

**প্রশ্ন ১:** A-এর shape (2, 3) আর B-এর shape (3, 4) হলে A @ B-এর shape কী?

**প্রশ্ন ২:** (3, 2) @ (3, 2) কি চলবে? কেন?

**প্রশ্ন ৩:** A-এর shape (4, 5) হলে `A.T`-এর shape কী?

**প্রশ্ন ৪:** [1, 2, 3] আর [4, 5, 6]-এর dot product কত?

**প্রশ্ন ৫:** Cosine similarity ১ আর ০ হলে দুই Vector সম্পর্কে কী বোঝা যায়?

**প্রশ্ন ৬:** Transformer-এ Positional Encoding কেন লাগে?

<details>
<summary>উত্তর দেখতে click করো</summary>

১. **(2, 4)।** মাঝের দুই সংখ্যা ৩ ও ৩ মিলেছে, আর ফল বাইরের দুই সংখ্যা।

২. **চলবে না।** মাঝের দুই সংখ্যা ২ ও ৩ মেলে না। (প্রথম Matrix-এর column আর দ্বিতীয়টার row সমান হতে হয়।)

৩. **(5, 4)।** Transpose-এ row আর column জায়গা বদলায়।

৪. **৩২।** 1×4 + 2×5 + 3×6 = 4 + 10 + 18 = 32।

৫. **১ হলে** দুই Vector প্রায় একই দিকে, অর্থাৎ অর্থ খুব কাছাকাছি। **০ হলে** তাদের মধ্যে কোনো সম্পর্ক নেই।

৬. Self-attention সব শব্দ একসাথে দেখে, কিন্তু শব্দের ক্রম নিজে থেকে জানে না। Positional Encoding শব্দের সাথে তার জায়গার তথ্য জুড়ে দেয়, তাই model ক্রম বুঝতে পারে।

</details>

নিচের cell-এ প্রশ্ন ১ থেকে ৪-এর উত্তর NumPy দিয়ে মিলিয়ে দেখা যায়।

In [16]:
print((np.zeros((2, 3)) @ np.zeros((3, 4))).shape)

try:
    np.zeros((3, 2)) @ np.zeros((3, 2))
except ValueError:
    print("চলে না, মাঝের সংখ্যা মেলেনি")

print(np.zeros((4, 5)).T.shape)

print(np.array([1, 2, 3]) @ np.array([4, 5, 6]))

(2, 4)
চলে না, মাঝের সংখ্যা মেলেনি
(5, 4)
32
